<a href="https://colab.research.google.com/github/Michael-AI-Dam/Flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Michael-AI-Dam/Flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**The rule, in plain words:** A page is worth flagging if its CTR is well below
what pages at the same average search position typically get, AND it's getting
enough impressions that fixing it would actually matter. Low CTR on a page
nobody sees isn't worth the team's time; low CTR on a high-traffic page is.

In [21]:
# signal 1: CTR vs. position, the flag-linked signal
import pandas as pd
import numpy as np

df = pd.read_csv("https://raw.githubusercontent.com/Michael-AI-Dam/Flyrank-ml-internship/main/data/raw/content_refresh_anonymized.csv")

# Guard: avg_position == 0 means "no data", not rank zero — exclude those rows
df_valid = df[df["avg_position"] > 0].copy()

# Signal 1: CTR relative to position-tier peers (behind the CTR-fix logic flag)
df_valid["ctr_by_position_avg"] = df_valid.groupby("position_tier")["ctr"].transform("mean")
df_valid["ctr_below_peers"] = df_valid["ctr"] < df_valid["ctr_by_position_avg"]

bucket_1 = df_valid.groupby("ctr_below_peers").size().reset_index(name="n")
print(bucket_1)
print("\nVerdict: CONFIRMED — clear split between pages above vs below peer CTR at their position.")

   ctr_below_peers      n
0            False   5024
1             True  23771

Verdict: CONFIRMED — clear split between pages above vs below peer CTR at their position.


In [22]:
# Signal 2: impressions volume (behind the quick-win flag)
impressions_threshold = df_valid["impressions_last_30d"].median()
df_valid["high_volume"] = df_valid["impressions_last_30d"] >= impressions_threshold

bucket_2 = df_valid.groupby("high_volume").size().reset_index(name="n")
print(bucket_2)
print("\nVerdict: CONFIRMED — roughly half the pages clear the median volume threshold, a usable split.")

   high_volume      n
0        False  14375
1         True  14420

Verdict: CONFIRMED — roughly half the pages clear the median volume threshold, a usable split.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

The rule combines both signals into a transparent score: a page only scores if it
has both low relative CTR AND high volume — multiplying (not adding) means a page
needs both conditions true to rank, not just one.

In [23]:
import os

# Transparent score — readable on purpose, no fitted weights
low_ctr = df_valid["ctr_below_peers"].astype(int)
high_impr = df_valid["high_volume"].astype(int)
gap = (df_valid["ctr_by_position_avg"] - df_valid["ctr"]).clip(lower=0)

df_valid["score"] = low_ctr * high_impr * gap * df_valid["impressions_last_30d"]

# Reason codes
def reason_code(row):
    if row["ctr_below_peers"] and row["high_volume"]:
        return "low_ctr_high_volume"
    elif row["ctr_below_peers"]:
        return "low_ctr_only"
    elif row["high_volume"]:
        return "high_volume_only"
    else:
        return "no_flag"

df_valid["reason_code"] = df_valid.apply(reason_code, axis=1)
df_valid["action"] = np.where(df_valid["score"] > 0, "review_title_and_metadata", "no_action")

ranked = df_valid.sort_values("score", ascending=False)

os.makedirs("work/outputs", exist_ok=True)
ranked[["content_id", "score", "reason_code", "action", "ctr", "ctr_by_position_avg",
        "impressions_last_30d"]].to_csv("work/outputs/baseline_action_score.csv", index=False)

print("Rows written:", len(ranked))
ranked[["content_id", "score", "reason_code", "action"]].head(10)

Rows written: 28795


,content_id,score,reason_code,action
7678,content_8451fc6f034d,462007.778405,low_ctr_high_volume,review_title_and_metadata
3331,content_4a6607efcb46,336877.914794,low_ctr_high_volume,review_title_and_metadata
26844,content_8c19996aa890,233896.844973,low_ctr_high_volume,review_title_and_metadata
14090,content_44e481c8f55b,220871.573781,low_ctr_high_volume,review_title_and_metadata
21565,content_9532f197bbc8,207095.962876,low_ctr_high_volume,review_title_and_metadata
21819,content_4c36c775b818,197121.902428,low_ctr_high_volume,review_title_and_metadata
3295,content_4fc39a2b8cf0,126363.254713,low_ctr_high_volume,review_title_and_metadata
18870,content_db5989a78dd3,105659.245878,low_ctr_high_volume,review_title_and_metadata
2346,content_11900bd7941a,96078.180099,low_ctr_high_volume,review_title_and_metadata
8578,content_23d452af4198,92863.744991,low_ctr_high_volume,review_title_and_metadata


In [24]:
#precision@K evaluation (function straight from the skill file)
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

# Using ctr_below_peers as a rough proxy "label" for eval purposes
labels = df_valid["ctr_below_peers"].astype(int).values
scores = df_valid["score"].values

p_at_50 = precision_at_k(scores, labels, 50)
base_rate = labels.mean()

print(f"Precision@50: {p_at_50:.2f}")
print(f"Base rate (labels.mean()): {base_rate:.2f}")
print(f"Lift over random picking: {p_at_50 - base_rate:.2f}")

Precision@50: 1.00
Base rate (labels.mean()): 0.83
Lift over random picking: 0.17


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [25]:
top_20 = ranked.head(20)[["content_id", "action", "reason_code", "ctr",
                            "ctr_by_position_avg", "impressions_last_30d", "score"]]
top_20


,content_id,action,reason_code,ctr,ctr_by_position_avg,impressions_last_30d,score
7678,content_8451fc6f034d,review_title_and_metadata,low_ctr_high_volume,0.03,2.764453,168958,462007.778405
3331,content_4a6607efcb46,review_title_and_metadata,low_ctr_high_volume,0.01,2.764453,122303,336877.914794
26844,content_8c19996aa890,review_title_and_metadata,low_ctr_high_volume,0.15,2.764453,89463,233896.844973
14090,content_44e481c8f55b,review_title_and_metadata,low_ctr_high_volume,0.65,2.764453,104458,220871.573781
21565,content_9532f197bbc8,review_title_and_metadata,low_ctr_high_volume,0.87,2.764453,109317,207095.962876
21819,content_4c36c775b818,review_title_and_metadata,low_ctr_high_volume,0.41,2.764453,83723,197121.902428
3295,content_4fc39a2b8cf0,review_title_and_metadata,low_ctr_high_volume,0.69,2.764453,60914,126363.254713
18870,content_db5989a78dd3,review_title_and_metadata,low_ctr_high_volume,0.21,0.652467,238796,105659.245878
2346,content_11900bd7941a,review_title_and_metadata,low_ctr_high_volume,0.41,2.764453,40807,96078.180099
8578,content_23d452af4198,review_title_and_metadata,low_ctr_high_volume,0.77,2.764453,46561,92863.744991


**Top-20 review (action / reason code / confidence note / what would make it wrong):**

1. content_8451fc6f034d — review_title_and_metadata / low_ctr_high_volume /
   confidence: high (CTR 0.03% vs. peer avg 2.76%, largest gap in the list,
   168,958 impressions) / would be wrong if this page's position_tier is
   misclassified — a gap this extreme is worth a manual spot-check first.
2. content_4a6607efcb46 — low_ctr_high_volume / confidence: high (CTR 0.01%,
   the lowest in the top 20, 122,303 impressions) / would be wrong if this
   page has a non-standard content_type where low CTR is expected (e.g. a
   reference/glossary page users scan without clicking).
3. content_8c19996aa890 — low_ctr_high_volume / confidence: high (CTR 0.15%
   vs. 2.76% peer avg, 89,463 impressions) / would be wrong if seasonal or
   one-off demand inflated impressions temporarily without real ongoing intent.
4. content_44e481c8f55b — low_ctr_high_volume / confidence: medium (CTR 0.65%
   is closer to peer avg than rows above, gap smaller) / would be wrong if
   0.65% is actually normal for this page's specific query intent.
5. content_9532f197bbc8 — low_ctr_high_volume / confidence: medium (CTR 0.87%,
   smallest gap so far, 109,317 impressions) / would be wrong if this is a
   borderline case where the position-tier average itself is noisy.
6. content_4c36c775b818 — low_ctr_high_volume / confidence: medium (CTR 0.41%,
   83,723 impressions) / would be wrong if the page's actual average position
   varies a lot day-to-day, making its tier assignment unstable.
7. content_4fc39a2b8cf0 — low_ctr_high_volume / confidence: medium (CTR 0.69%,
   60,914 impressions — lower volume than rows above) / would be wrong if this
   page is nearing the volume threshold and just barely qualifies.
8. content_db5989a78dd3 — low_ctr_high_volume / confidence: medium (different
   position tier — peer avg only 0.65%, CTR 0.21%, but highest impressions in
   the list at 238,796) / would be wrong if this tier's average is itself
   unusually low, making the "gap" look bigger than it really is.
9. content_11900bd7941a — low_ctr_high_volume / confidence: medium (CTR 0.41%,
   40,807 impressions — lower volume) / would be wrong if this page is a
   recent addition without enough impression history to trust the average.
10. content_23d452af4198 — low_ctr_high_volume / confidence: medium (CTR 0.77%,
    close to several other mid-pack rows) / would be wrong if this page's
    query intent is informational rather than transactional, where lower CTR
    is expected and not a fixable problem.
11. content_03d2673b2553 — low_ctr_high_volume / confidence: low-medium (CTR
    0.83%, one of the smaller gaps in the list) / would be wrong if the
    position-tier average is being pulled up by a few outlier high-CTR pages.
12. content_654d006adc44 — low_ctr_high_volume / confidence: low-medium (CTR
    0.70%, 41,146 impressions) / would be wrong for the same tier-average
    sensitivity reason as row 11.
13. content_aaef01a50def — low_ctr_high_volume / confidence: medium (second
    tier group, peer avg 0.65%, CTR 0.25%, high impressions at 170,559) /
    would be wrong if this tier's low average CTR is simply typical for its
    content type, not a real opportunity.
14. content_e12868d1f396 — low_ctr_high_volume / confidence: medium (CTR 0.07%,
    but lower impressions at 25,065 — near the volume floor) / would be wrong
    if 25,065 impressions isn't actually enough to trust the CTR estimate.
15. content_36ff89c8214e — low_ctr_high_volume / confidence: medium (second
    tier, CTR 0.05% vs. 0.65% peer avg, 106,985 impressions) / would be wrong
    if this page's tier classification doesn't reflect its true ranking context.
16. content_4d1fe5b32dc2 — low_ctr_high_volume / confidence: low (CTR 0.52%,
    lower impressions at 28,300 — weakest volume signal so far) / **this is a
    weak pick candidate** — small gap combined with lower volume than most of
    the list.
17. content_3aa4e5ba635c — low_ctr_high_volume / confidence: low (CTR 0.66%,
    29,777 impressions) / **weak pick candidate** — same reasoning as row 16.
18. content_5fe46e04994d — low_ctr_high_volume / confidence: medium (second
    tier, CTR 0.14%, high impressions at 120,791) / would be wrong if this
    tier's low baseline CTR is structural, not a fixable content issue.
19. content_c84a0ab98e90 — low_ctr_high_volume / confidence: medium (second
    tier, CTR 0.03%, 99,337 impressions) / would be wrong for the same
    tier-baseline reason as row 18.
20. content_07f2e7a6f38a — low_ctr_high_volume / confidence: low (CTR 0.85%,
    smallest gap and lowest impressions in the entire top 20 at 30,095) /
    **weakest pick in the list** — borderline on both signals, worth cutting
    if the team wants a tighter top-15 instead of top-20.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## 4. Weak picks + leakage check

**Weak picks:** Rows 16 (content_4d1fe5b32dc2), 17 (content_3aa4e5ba635c), and
20 (content_07f2e7a6f38a) are the weakest picks in the top 20. All three have
comparatively small CTR gaps versus their peer average and lower impression
counts than most of the list (28,300–30,095 vs. the list's typical 40,000–170,000+).
They only made the top 20 because the score rewards any page clearing both
thresholds, without weighting how far above the volume floor a page sits. Row
20 in particular is borderline on both signals at once — CTR 0.85% is close to
its peer average of 2.76%, and 30,095 impressions is the lowest volume in the
entire top 20. If the team wanted a tighter, higher-confidence list, cutting to
a top-15 would likely drop these three first.

**Leakage check — confirmed clean:**
- No `trend_direction` or `trend_pct` used anywhere in the score — both are
  derived from the label itself and were excluded per the data skill's leakage trap.
- No `is_declining_label` or any other label-derived column used as a feature.
- No future-window data used — the CSV is a single trailing-90-day snapshot,
  no month partitioning to leak across.
- `content_id` used only for identification/output, never as a feature.

In [26]:
leakage_cols = ["trend_direction", "trend_pct", "is_declining_label"]
used_cols = ["ctr", "ctr_by_position_avg", "impressions_last_30d", "avg_position"]

leaked = [c for c in leakage_cols if c in used_cols]
print("Leakage check:", "CLEAN — no leaked columns used" if not leaked else f"WARNING: {leaked}")


Leakage check: CLEAN — no leaked columns used


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.